# Recherche systématique de règles de construction

Ce carnet applique la phase 0 de la section IX de `research/plan_projet.md`, commitée avant tout calcul. Il construit 900 règles, soit quatre sélections, neuf pondérations et vingt-cinq protections. Il les juge sur les critères du bloc 3 face à `T1S`, les place dans la distribution de 100 000 portefeuilles tirés au hasard, puis soumet le gagnant à trois contrôles. Les fonctions viennent de `src/regles.py`, testées dans `tests/test_regles.py`.

L'univers reste celui choisi avec les rapports de 2026 : tout ce qui suit compare des règles à l'intérieur de cet univers et ne décrit aucune performance réalisable.

In [1]:
import sys, json, gzip, time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from src import regles, risque, couverture as cv
from src.collecter_couverture import verifier
from src.empreintes import empreinte

SORTIE, BRUT = RACINE / "data" / "processed", RACINE / "data" / "raw"
verifier(RACINE)
manifeste_fonda = json.loads((BRUT / "fondamentaux" / "manifeste.json").read_text(encoding="utf-8"))
assert empreinte(BRUT / "fondamentaux" / "faits_10k.csv") == manifeste_fonda["sha256"], "faits comptables modifies"

rv = pd.read_csv(SORTIE / "rendements_valorisation.csv", index_col=0)
dv = pd.read_csv(SORTIE / "dividendes.csv", index_col=0).reindex_like(rv)
R = rv + dv.fillna(0.0).where(rv.notna())          # rendement total, dividende reinvesti au detachement
dates = R.index
indice_prix = (1 + rv.fillna(0.0)).cumprod().where(rv.notna().cumsum() > 0)

appartenance = pd.read_csv(SORTIE / "appartenance.csv", dtype=str)
GROUPES = {"P4": "acheteurs", "P5": "vendeurs a la chaine", "P6": "electricite", "P7": "equipement",
           "P8": "immobilier", "P9": "fournisseurs technologiques", "P10": "autres fournisseurs"}
p1 = appartenance[appartenance.portefeuille == "P1"]
entreprise_de = dict(zip(p1.titre, p1.cik))
groupe_de_cik = {r.cik: GROUPES[r.portefeuille] for r in appartenance[appartenance.portefeuille.isin(GROUPES)].itertuples()}
groupe_de = {t: groupe_de_cik[c] for t, c in entreprise_de.items()}
titre_de = p1.groupby("cik").titre.first().to_dict()
assert set(groupe_de_cik) == set(entreprise_de.values()), "les sept groupes doivent partitionner P1"

taux = pd.read_csv(BRUT / "taux_sans_risque.csv", index_col=0)["taux_annuel_pct"].reindex(dates).ffill()
rf = (1 + taux / 100) ** (1 / 252) - 1
spy = pd.read_csv(SORTIE / "valeurs_comparaisons.csv", index_col=0)["SPY"]
r_spy = spy.pct_change()
r_inter = pd.read_csv(BRUT / "couverture" / "VFITX.csv", index_col=0)["Adj Close"].pct_change().reindex(dates)
sptr = pd.read_csv(BRUT / "benchmarks" / "SP500TR.csv")
sptr.index = sptr.date.str[:10]
pput = pd.read_csv(BRUT / "couverture" / "PPUT_cboe.csv")
pput.index = pd.to_datetime(pput.DATE, format="%m/%d/%Y").dt.strftime("%Y-%m-%d")
niveaux = pd.concat([pput.PPUT, sptr.Close], axis=1, keys=["pput", "sptr"]).dropna()
jambe = (niveaux.pput.pct_change() - niveaux.sptr.pct_change()).reindex(dates)
t1s = pd.read_csv(SORTIE / "temoin_valeurs.csv", index_col=0)["T1S_reeq"]
faits = pd.read_csv(BRUT / "fondamentaux" / "faits_10k.csv", dtype={"cik": str})

annuel = sorted(cv.premieres_seances_annee(dates))
fins = cv.fins_de_mois(dates)
DEBUT = "2011-01-03"
print(R.shape, "| jambe d'options absente sur la fenetre :", int(jambe.loc[DEBUT:].isna().sum()),
      "| VFITX absent :", int(r_inter.loc[DEBUT:].isna().sum()))
print("rendement annuel moyen de la jambe d'options : %.2f %%" % (100 * jambe.loc[DEBUT:].mean() * 252))

(6709, 135) | jambe d'options absente sur la fenetre : 3 | VFITX absent : 0
rendement annuel moyen de la jambe d'options : -3.93 %


## 1. Éligibilité et contrôle du moteur

Un titre est éligible à une date de rééquilibrage s'il a un cours ce jour-là et au moins 252 séances d'historique. Premier contrôle : la règle S0-W1 sans protection, construite depuis 2000, doit rester proche de `P1` rééquilibré de l'étape 2. Elle n'en diffère que par trois conventions : réinvestissement des dividendes au détachement, entrée des titres en janvier seulement, et 252 séances d'historique exigées.

In [2]:
eligibles = {}
for d in annuel:
    passe = R.loc[:d]
    ok = passe.iloc[-1].notna() & (passe.iloc[:-1].notna().sum() >= 252)
    eligibles[d] = list(ok[ok].index)
print({d[:4]: len(v) for d, v in eligibles.items()})
DEBUT_LONG = next(d for d in annuel if eligibles[d])
print("DEBUT_LONG :", DEBUT_LONG)

def construire_actions(selection, ponderation, fenetre, debut, notes_par_date):
    cibles = {}
    for d in [x for x in annuel if x >= debut]:
        titres = eligibles[d]
        if selection != "S0":
            notes = notes_par_date[d]
            retenues = set(regles.selectionner(selection, notes, groupe_de_cik))
            titres = [t for t in titres if entreprise_de[t] in retenues]
        passe = R.loc[:d].iloc[:-1].iloc[-(fenetre or 252):]
        cibles[d] = regles.ponderer(ponderation, passe, titres, groupe_de, entreprise_de)
    v, frais = regles.simuler(R.loc[debut:], cibles)
    return v, frais, cibles

controle, _, _ = construire_actions("S0", "W1", None, DEBUT_LONG, None)
p1r = pd.read_csv(SORTIE / "valeurs_portefeuilles.csv", index_col=0)["P1_reeq"].loc[DEBUT_LONG:]
print("S0-W1 depuis 2000 : %.2f %% par an, P1 reeq : %.2f %% ; volatilite %.2f %% contre %.2f %% ; correlation quotidienne %.4f"
      % (100 * risque.annualiser_rendement(controle), 100 * risque.annualiser_rendement(p1r),
         100 * risque.volatilite(controle.pct_change().dropna()), 100 * risque.volatilite(p1r.pct_change().dropna()),
         controle.pct_change().corr(p1r.pct_change())))

{'2000': 0, '2001': 90, '2002': 93, '2003': 94, '2004': 97, '2005': 98, '2006': 102, '2007': 102, '2008': 104, '2009': 106, '2010': 106, '2011': 107, '2012': 110, '2013': 113, '2014': 115, '2015': 116, '2016': 120, '2017': 122, '2018': 125, '2019': 126, '2020': 127, '2021': 128, '2022': 130, '2023': 131, '2024': 132, '2025': 132, '2026': 133}
DEBUT_LONG : 2001-01-02


S0-W1 depuis 2000 : 18.13 % par an, P1 reeq : 18.94 % ; volatilite 21.58 % contre 21.57 % ; correlation quotidienne 0.9987


## 2. Les notes fondamentales, date par date

Pour chaque premier jour de bourse depuis 2011, les cinq critères sont calculés sur le dernier rapport annuel déposé avant cette date, puis convertis en rangs centiles dans chacun des sept groupes.

In [3]:
notes_par_date, couverture_criteres = {}, []
for d in [x for x in annuel if x >= DEBUT]:
    fonda = regles.fondamentaux_a_date(faits, d)
    crit = regles.criteres(fonda, indice_prix, titre_de, d)
    ciks = {entreprise_de[t] for t in eligibles[d]}
    notes_par_date[d] = regles.notes_par_groupe(crit.reindex(sorted(ciks)), groupe_de_cik, ciks)
    couverture_criteres.append(crit.reindex(sorted(ciks)).notna().mean().rename(d[:4]))
couverture_criteres = pd.DataFrame(couverture_criteres)
print("part des entreprises eligibles ou chaque critere est disponible :")
print(couverture_criteres.round(2).to_string())

part des entreprises eligibles ou chaque critere est disponible :
      profitabilite  qualite  croissance_actif  rendement_benefice  solidite
2011           0.42     0.53              0.62                0.53      0.60
2012           0.61     0.76              0.87                0.77      0.86
2013           0.64     0.78              0.93                0.77      0.93
2014           0.67     0.82              0.94                0.81      0.94
2015           0.66     0.80              0.94                0.80      0.94
2016           0.66     0.80              0.94                0.80      0.94
2017           0.68     0.83              0.95                0.82      0.95
2018           0.69     0.81              0.94                0.81      0.94
2019           0.68     0.82              0.96                0.82      0.96
2020           0.71     0.85              0.97                0.84      0.96
2021           0.72     0.84              0.98                0.84      0.97
2022      

## 3. Les 36 règles d'actions, puis les 900 règles

In [4]:
SELECTIONS = ["S0", "S1", "S2", "S3"]
PONDERATIONS = [("W1", None)] + [(w, f) for w in ["W2", "W3", "W4", "W5"] for f in (252, 504)]
debut_chrono = time.time()
actions, cibles_actions = {}, {}
for s in SELECTIONS:
    for w, f in PONDERATIONS:
        nom = f"{s}-{w}" + (f"-{f}" if f else "")
        v, frais, cibles = construire_actions(s, w, f, DEBUT, notes_par_date)
        actions[nom], cibles_actions[nom] = v, cibles
print(len(actions), "regles d'actions en %.0f s" % (time.time() - debut_chrono))

def protections(v):
    r = v.pct_change()
    beta = cv.beta_glissant(r, r_spy.reindex(r.index))
    sorties = {"P0": v}
    for cible in (0.10, 0.12, 0.15, 0.18):
        for n in (21, 63):
            for reste, r_reste in [("liq", rf), ("obl", r_inter)]:
                sorties[f"VT{round(100 * cible)}-{n}-{reste}"] = regles.pilotage_volatilite(r, r_reste.reindex(r.index), cible, n, fins)
    poches = pd.DataFrame({"actions": r, "liq": rf.reindex(r.index), "obl": r_inter.reindex(r.index)})
    jours_reeq = {d for d in annuel if d >= r.index[0]}
    for part in (0.8, 0.6):
        for reste, repartition in [("liq", {"liq": 1.0}), ("obl", {"obl": 1.0}), ("mixte", {"liq": 0.5, "obl": 0.5})]:
            w = pd.Series({"actions": part, "liq": 0.0, "obl": 0.0})
            for k, x in repartition.items():
                w[k] = (1 - part) * x
            sorties[f"MX{round(100 * part)}-{reste}"] = regles.simuler(poches, {d: w for d in jours_reeq})[0]
    for part in (0.5, 1.0):
        sorties[f"PUT{round(100 * part)}"] = regles.protection_indice(r, jambe, beta, part, fins)
    return sorties

regles_valeurs = {}
for nom, v in actions.items():
    for p, serie in protections(v).items():
        regles_valeurs[f"{nom}|{p}"] = serie
regles_valeurs = pd.DataFrame(regles_valeurs)
print(regles_valeurs.shape, "| valeurs non positives :", int((regles_valeurs <= 0).sum().sum()),
      "| absentes :", int(regles_valeurs.isna().sum().sum()))

36 regles d'actions en 210 s


(3942, 900) | valeurs non positives : 0 | absentes : 0


## 4. Mesures et critères du bloc 3

Les critères comparent chaque règle à `T1S` rééquilibré, rebasé au 3 janvier 2011, sur la même fenêtre.

In [5]:
def mesures(v):
    r = v.pct_change().dropna()
    m = r_spy.reindex(r.index)
    ok = m.notna()
    return {"annualise": risque.annualiser_rendement(v), "volatilite": risque.volatilite(r),
            "repli_max": risque.drawdown(v)["repli_max"], "es99": risque.perte_au_dela(r, 0.99),
            "sharpe": risque.sharpe(r, rf), "beta_spy": float(np.polyfit(m[ok], r[ok], 1)[0])}

tableau = pd.DataFrame({c: mesures(regles_valeurs[c]) for c in regles_valeurs}).T
temoin = pd.Series(mesures(t1s.loc[DEBUT:]))
tableau["acceptable"] = ((tableau.repli_max >= temoin.repli_max) & (tableau.es99 <= temoin.es99)
                         & (tableau.annualise >= temoin.annualise))
parties = tableau.index.to_series().str.split(r"[-|]", regex=False)
tableau["selection"] = tableau.index.str[:2]
tableau["ponderation"] = tableau.index.str.split("|").str[0].str[3:]
tableau["protection"] = tableau.index.str.split("|").str[1]
tableau["famille_protection"] = tableau.protection.str.extract(r"^([A-Z]+)")[0]
print("T1S sur la fenetre :", temoin.round(4).to_dict())
print("regles acceptables : %d sur %d" % (tableau.acceptable.sum(), len(tableau)))
print(tableau.groupby("famille_protection").acceptable.agg(["sum", "count"]).to_string())
print()
print("chaque critere pris seul :",
      {"repli": int((tableau.repli_max >= temoin.repli_max).sum()), "es99": int((tableau.es99 <= temoin.es99).sum()),
       "rendement": int((tableau.annualise >= temoin.annualise).sum())})
acceptables = tableau[tableau.acceptable].sort_values("sharpe", ascending=False)
print()
print(acceptables.head(15)[["annualise", "volatilite", "repli_max", "es99", "sharpe"]].round(4).to_string())
gagnant = acceptables.index[0] if len(acceptables) else tableau.sharpe.idxmax()
print("\nregle retenue avant controles :", gagnant, "(acceptable)" if len(acceptables) else "(aucune acceptable : meilleur Sharpe de la grille)")

T1S sur la fenetre : {'annualise': 0.1609, 'volatilite': 0.1733, 'repli_max': -0.3821, 'es99': 0.0442, 'sharpe': 0.8604, 'beta_spy': 0.9577}
regles acceptables : 102 sur 900
                    sum  count
famille_protection            
MX                   12    216
P                    12     36
PUT                  15     72
VT                   63    576

chaque critere pris seul : {'repli': 875, 'es99': 884, 'rendement': 118}

                       annualise  volatilite  repli_max    es99  sharpe
S0-W3-504|VT15-21-obl     0.1700      0.1453    -0.2772  0.0349  1.0496
S0-W3-252|VT15-21-obl     0.1691      0.1453    -0.2768  0.0351  1.0438
S0-W3-252|PUT50           0.1802      0.1564    -0.2712  0.0347  1.0405
S0-W1|VT15-21-obl         0.1732      0.1500    -0.2714  0.0355  1.0393
S0-W1|PUT50               0.1926      0.1689    -0.2563  0.0368  1.0377
S0-W3-504|PUT50           0.1803      0.1573    -0.2714  0.0348  1.0361
S0-W3-252|MX80-obl        0.1652      0.1432    -0.2857  0.03

## 5. Les 100 000 portefeuilles aléatoires

In [6]:
debut_chrono = time.time()
aleatoires = {}
for nom, depart in [("fenetre_2011", DEBUT), ("depuis_2000", DEBUT_LONG)]:
    elig = {d: v for d, v in eligibles.items() if d >= depart}
    aleatoires[nom] = regles.portefeuilles_aleatoires(R.loc[depart:], elig, 100_000, 20260929, rf)
    print(nom, "en %.0f s" % (time.time() - debut_chrono))
    print(aleatoires[nom].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(4).to_string())
    print()

hasard = aleatoires["fenetre_2011"]
def centile(colonne, valeur, plus_haut_mieux=True):
    x = hasard[colonne]
    return float((x < valeur).mean() if plus_haut_mieux else (x > valeur).mean())
tableau["centile_sharpe"] = [centile("sharpe", s) for s in tableau.sharpe]
sans = tableau.protection == "P0"
for col, sens in [("volatilite", False), ("repli_max", True), ("es99", False)]:
    tableau.loc[sans, f"centile_{col}"] = [centile(col, x, sens) for x in tableau.loc[sans, col]]
print("centile de Sharpe des regles, face au hasard :")
print(tableau.groupby("famille_protection").centile_sharpe.describe().round(3).to_string())
print()
print(tableau.loc[sans, ["sharpe", "centile_sharpe", "centile_volatilite", "centile_repli_max", "centile_es99"]].round(3).to_string())
print("\nT1S : centile de Sharpe %.3f" % centile("sharpe", temoin.sharpe))

fenetre_2011 en 47 s
            taille    annualise   volatilite       sharpe    repli_max         es99
count  100000.0000  100000.0000  100000.0000  100000.0000  100000.0000  100000.0000
mean       72.5137       0.2105       0.2019       0.9722      -0.3694       0.0488
std        36.3637       0.0223       0.0143       0.0730       0.0191       0.0031
min        10.0000       0.1019       0.1570       0.4440      -0.7052       0.0394
1%         11.0000       0.1547       0.1777       0.7470      -0.4284       0.0437
5%         16.0000       0.1761       0.1848       0.8412      -0.4001       0.0452
50%        73.0000       0.2097       0.1995       0.9788      -0.3682       0.0483
95%       129.0000       0.2465       0.2282       1.0775      -0.3419       0.0543
99%       134.0000       0.2758       0.2536       1.1315      -0.3251       0.0600
max       135.0000       0.4530       0.3738       1.3845      -0.2469       0.0974



depuis_2000 en 136 s
            taille    annualise   volatilite       sharpe    repli_max         es99
count  100000.0000  100000.0000  100000.0000  100000.0000  100000.0000  100000.0000
mean       72.5137       0.1792       0.2229       0.7739      -0.5385       0.0528
std        36.3637       0.0165       0.0145       0.0565       0.0333       0.0028
min        10.0000       0.0882       0.1717       0.3854      -0.8915       0.0418
1%         11.0000       0.1354       0.1984       0.5969      -0.6533       0.0481
5%         16.0000       0.1530       0.2055       0.6714      -0.5969       0.0495
50%        73.0000       0.1790       0.2204       0.7793      -0.5344       0.0523
95%       129.0000       0.2053       0.2503       0.8549      -0.4954       0.0580
99%       134.0000       0.2249       0.2759       0.8950      -0.4716       0.0635
max       135.0000       0.3173       0.3979       1.0327      -0.3413       0.0877

centile de Sharpe des regles, face au hasard :
       

## 6. Les trois contrôles du gagnant

Bootstrap par blocs d'un trimestre de l'écart de Sharpe avec `T1S`, Deflated Sharpe Ratio sur 900 essais, probabilité de surapprentissage sur seize blocs, et test de White en complément.

In [7]:
def excedents(v):
    r = v.pct_change().dropna()
    return (r - rf.reindex(r.index)).to_numpy()

sharpe_q = lambda x: x.mean() / x.std(ddof=1) * np.sqrt(252)
a, b = excedents(regles_valeurs[gagnant]), excedents(t1s.loc[DEBUT:])
controle_bootstrap = risque.ecart_bootstrap(a, b, sharpe_q, 63, 2000)
print("ecart de Sharpe avec T1S :", {k: round(v, 4) for k, v in controle_bootstrap.items()})

E = np.column_stack([excedents(regles_valeurs[c]) for c in regles_valeurs])
sr_quotidiens = E.mean(0) / E.std(0, ddof=1)
x = E[:, list(regles_valeurs.columns).index(gagnant)]
asym = float(pd.Series(x).skew())
aplat = float(pd.Series(x).kurt()) + 3
dsr, sr0 = regles.sharpe_deflate(x.mean() / x.std(ddof=1), sr_quotidiens.var(ddof=1), E.shape[1], len(x), asym, aplat)
print("Deflated Sharpe Ratio : %.4f (Sharpe annuel du gagnant %.3f, seuil du hasard %.3f)"
      % (dsr, x.mean() / x.std(ddof=1) * np.sqrt(252), sr0 * np.sqrt(252)))

pbo, logits = regles.probabilite_surapprentissage(E, blocs=16)
print("probabilite de surapprentissage : %.3f sur %d combinaisons" % (pbo, len(logits)))

indices = risque.indices_blocs(len(b), 63, 1000, graine=1)
ecart_max, p_white = regles.test_white(E, b, indices)
print("test de White : ecart de Sharpe quotidien maximal %.4f, probabilite %.3f" % (ecart_max, p_white))

ecart de Sharpe avec T1S : {'ecart': 0.1892, 'bas': -0.0494, 'haut': 0.4012, 'p_bilaterale': 0.129}


Deflated Sharpe Ratio : 0.9985 (Sharpe annuel du gagnant 1.050, seuil du hasard 0.283)


probabilite de surapprentissage : 0.594 sur 12870 combinaisons


test de White : ecart de Sharpe quotidien maximal 0.0124, probabilite 0.455


## 7. Les quatre attentes écrites avant calcul

In [8]:
print("1. Deflated Sharpe Ratio du gagnant > 0,95 ?", dsr > 0.95)

print("\n2. Part du risque de P5 et P9 au 4 septembre 2026, selon la ponderation (selection S0) :")
fenetre_finale = R.iloc[-252:]
parts = {}
for nom in [n for n in actions if n.startswith("S0-")]:
    derniere = cibles_actions[nom][annuel[-1]]
    poids_fin = derniere * (1 + fenetre_finale[derniere.index].loc[annuel[-1]:].iloc[1:].fillna(0)).prod()
    poids_fin = poids_fin / poids_fin.sum()
    bloc = fenetre_finale[poids_fin.index].fillna(0).to_numpy()
    cov = np.cov(bloc, rowvar=False)
    contrib, vol = risque.contributions_au_risque(poids_fin.to_numpy(), cov)
    groupes = pd.Series([groupe_de[t] for t in poids_fin.index])
    part = pd.Series(contrib / vol).groupby(groupes).sum()
    parts[nom] = part
parts = pd.DataFrame(parts).T
parts["P5_plus_P9"] = parts["vendeurs a la chaine"] + parts["fournisseurs technologiques"]
print(parts.round(3).to_string())

print("\n3. Regles acceptables par famille de protection :")
print(tableau[tableau.acceptable].famille_protection.value_counts().to_string() if tableau.acceptable.any() else "aucune")

print("\n4. Selection fondamentale contre S0, a ponderation et protection egales (bootstrap de l'ecart de Sharpe) :")
lignes = []
for c in regles_valeurs.columns:
    s = c[:2]
    if s == "S0":
        continue
    reference = "S0" + c[2:]
    res = risque.ecart_bootstrap(excedents(regles_valeurs[c]), excedents(regles_valeurs[reference]), sharpe_q, 63, 500)
    lignes.append({"regle": c, "selection": s, **res})
paires = pd.DataFrame(lignes)
paires["significatif"] = paires.p_bilaterale < 0.05
print(paires.groupby("selection").agg(paires=("regle", "size"), ecart_median=("ecart", "median"),
      significatifs_hausse=("significatif", lambda s: int((s & (paires.loc[s.index, "ecart"] > 0)).sum())),
      significatifs_baisse=("significatif", lambda s: int((s & (paires.loc[s.index, "ecart"] < 0)).sum()))).round(4).to_string())

1. Deflated Sharpe Ratio du gagnant > 0,95 ? True

2. Part du risque de P5 et P9 au 4 septembre 2026, selon la ponderation (selection S0) :
           acheteurs  autres fournisseurs  electricite  equipement  fournisseurs technologiques  immobilier  vendeurs a la chaine  P5_plus_P9
S0-W1          0.018                0.037        0.026       0.186                        0.154       0.015                 0.564       0.719
S0-W2-252      0.022                0.056        0.052       0.230                        0.148       0.028                 0.464       0.612
S0-W2-504      0.022                0.057        0.052       0.233                        0.144       0.027                 0.465       0.608
S0-W3-252      0.056                0.075        0.084       0.162                        0.238       0.117                 0.269       0.507
S0-W3-504      0.056                0.076        0.080       0.161                        0.239       0.110                 0.277       0.516
S0-W4-25

           paires  ecart_median  significatifs_hausse  significatifs_baisse
selection                                                                  
S1            225       -0.0663                     0                    79
S2            225       -0.0888                     0                   123
S3            225       -0.0638                     0                    47


## 8. Lecture secondaire depuis 2000, pour les règles sans sélection fondamentale

In [9]:
secondaire = {}
for w, f in PONDERATIONS:
    nom = "S0-" + w + (f"-{f}" if f else "")
    v, _, _ = construire_actions("S0", w, f, DEBUT_LONG, None)
    for p, serie in protections(v).items():
        secondaire[f"{nom}|{p}"] = serie
secondaire = pd.DataFrame(secondaire)
tab2 = pd.DataFrame({c: mesures(secondaire[c]) for c in secondaire}).T
temoin2 = pd.Series(mesures(t1s.loc[DEBUT_LONG:]))
tab2["acceptable"] = ((tab2.repli_max >= temoin2.repli_max) & (tab2.es99 <= temoin2.es99) & (tab2.annualise >= temoin2.annualise))
hasard2 = aleatoires["depuis_2000"]
tab2["centile_sharpe"] = [float((hasard2.sharpe < s).mean()) for s in tab2.sharpe]
print("T1S depuis 2000 :", temoin2.round(4).to_dict())
print("acceptables : %d sur %d" % (tab2.acceptable.sum(), len(tab2)))
print(tab2.sort_values("sharpe", ascending=False).head(10)[["annualise", "volatilite", "repli_max", "es99", "sharpe", "acceptable", "centile_sharpe"]].round(4).to_string())

T1S depuis 2000 : {'annualise': 0.1689, 'volatilite': 0.1951, 'repli_max': -0.5044, 'es99': 0.0488, 'sharpe': 0.809, 'beta_spy': 0.9646}
acceptables : 0 sur 225
                       annualise  volatilite  repli_max    es99  sharpe  acceptable  centile_sharpe
S0-W3-504|VT10-21-obl     0.1305      0.1124    -0.2352  0.0261  0.9933       False          0.9998
S0-W3-252|VT10-21-obl     0.1294      0.1125    -0.2347  0.0262  0.9838       False          0.9997
S0-W4-504|VT10-21-obl     0.1266      0.1112    -0.2266  0.0257  0.9717       False          0.9995
S0-W3-504|VT12-21-obl     0.1428      0.1281    -0.2870  0.0292  0.9709       False          0.9995
S0-W4-252|VT10-21-obl     0.1257      0.1111    -0.2262  0.0258  0.9651       False          0.9994
S0-W3-252|VT12-21-obl     0.1416      0.1283    -0.2865  0.0294  0.9622       False          0.9994
S0-W3-504|MX60-obl        0.1279      0.1155    -0.2949  0.0268  0.9503       False          0.9989
S0-W3-252|MX60-obl        0.1279      0

## 9. Sorties et manifeste

In [10]:
def ecrire_gz(df, nom):
    chemin = SORTIE / nom
    chemin.write_bytes(gzip.compress(df.to_csv(float_format="%.10g", lineterminator="\n").encode("utf-8"), mtime=0))

ecrire_gz(regles_valeurs, "regles_valeurs.csv.gz")
tableau.to_csv(SORTIE / "regles_mesures.csv", encoding="utf-8", lineterminator="\n")
tab2.to_csv(SORTIE / "regles_mesures_depuis_2000.csv", encoding="utf-8", lineterminator="\n")
for nom, df in aleatoires.items():
    ecrire_gz(df, f"aleatoires_{nom}.csv.gz")
paires.to_csv(SORTIE / "regles_selection_contre_s0.csv", index=False, encoding="utf-8", lineterminator="\n")
parts.to_csv(SORTIE / "regles_parts_de_risque.csv", encoding="utf-8", lineterminator="\n")
controles = {"gagnant": gagnant, "acceptables": int(tableau.acceptable.sum()), "bootstrap_t1s": controle_bootstrap,
             "sharpe_deflate": dsr, "seuil_hasard_sharpe_annuel": sr0 * np.sqrt(252), "pbo": pbo,
             "white_ecart_max_quotidien": ecart_max, "white_p": p_white, "essais": int(E.shape[1])}
sorties = ["regles_valeurs.csv.gz", "regles_mesures.csv", "regles_mesures_depuis_2000.csv", "aleatoires_fenetre_2011.csv.gz",
           "aleatoires_depuis_2000.csv.gz", "regles_selection_contre_s0.csv", "regles_parts_de_risque.csv"]
(SORTIE / "regles_manifest.json").write_text(json.dumps({
    "produit_par": "src/recherche_regles.ipynb", "decisions": "research/plan_projet.md, section IX",
    "controles": controles,
    "entrees_sha256": {"faits_10k.csv": manifeste_fonda["sha256"], "rendements_valorisation.csv": empreinte(SORTIE / "rendements_valorisation.csv"),
                       "PPUT_cboe.csv": empreinte(BRUT / "couverture" / "PPUT_cboe.csv"), "temoin_valeurs.csv": empreinte(SORTIE / "temoin_valeurs.csv")},
    "code_sha256": {"src/regles.py": empreinte(RACINE / "src" / "regles.py")},
    "sorties_sha256": {s: empreinte(SORTIE / s) for s in sorties}}, ensure_ascii=False, indent=2, default=float) + "\n", encoding="utf-8")
print(json.dumps(controles, indent=1, default=float))

{
 "gagnant": "S0-W3-504|VT15-21-obl",
 "acceptables": 102,
 "bootstrap_t1s": {
  "ecart": 0.18924118411443858,
  "bas": -0.04943327802769187,
  "haut": 0.40120603938888294,
  "p_bilaterale": 0.129
 },
 "sharpe_deflate": 0.9985162391383465,
 "seuil_hasard_sharpe_annuel": 0.283219547057819,
 "pbo": 0.5944055944055944,
 "white_ecart_max_quotidien": 0.012384024552124055,
 "white_p": 0.455,
 "essais": 900
}
